# How Many Zoos Are in the Dataset?

This notebook isolates **zoos specifically** (excluding aquariums and wildlife preserves) and reports the total count.

Self-contained: only requires `museums.csv` in the same folder. Run **Kernel → Restart Kernel and Run All Cells**.

## Setup

In [1]:
import pandas as pd
df = pd.read_csv("museums.csv", low_memory=False)
print(f"Loaded {len(df):,} museums")

Loaded 33,072 museums


## Step 1: Why we need both a category filter and a name filter

The IMLS dataset bundles zoos, aquariums, and wildlife conservation into one combined `Museum Type` category — there's no "zoo" tag on its own. To isolate zoos specifically, we'll combine two filters:

1. The IMLS category (`ZOO, AQUARIUM, OR WILDLIFE CONSERVATION`) — narrows to ~564 records.
2. A name match for `\bzoo` — keeps only records whose name actually contains the word "zoo" (catches "Zoo", "Zoos", "Zoological" but excludes accidental matches like "Kalamazoo").

## Step 2: Filter by category

In [2]:
category_mask = df["Museum Type"] == "ZOO, AQUARIUM, OR WILDLIFE CONSERVATION"
category_subset = df[category_mask]
print(f"Museums in the ZOO/AQUARIUM/WILDLIFE category: {len(category_subset)}")

Museums in the ZOO/AQUARIUM/WILDLIFE category: 564


## Step 3: Filter by name to isolate zoos

In [3]:
# \bzoo with a word boundary catches:
#   ✓ ZOO, ZOOS, ZOOLOGICAL, ZOOLOGY
#   ✗ KALAMAZOO  (no word boundary before 'zoo' inside the word)
name_mask = category_subset["Museum Name"].fillna("").str.contains(
    r"\bzoo", case=False, regex=True)

zoos = category_subset[name_mask]
print(f"Zoos (in category AND name contains 'zoo'): {len(zoos)}")

Zoos (in category AND name contains 'zoo'): 258


### Spot-check: sample of matched zoos

In [4]:
zoos["Museum Name"].sample(15, random_state=1).tolist()

['OKLAHOMA CITY ZOOLOGICAL AND BOTANICAL PARK',
 'SMITHSONIAN NATIONAL ZOOLOGICAL PARK',
 'OKLAHOMA ZOO',
 'UTAH ZOOLOGICAL SOCIETY',
 'AMARILLO ZOOLOGICAL SOCIETY',
 'LITTLE RIVER ZOO',
 'BUFFALO ZOOLOGICAL GARDENS',
 'SEQUOIA PARK ZOO',
 "OMAHA'S HENRY DOORLY ZOO & AQUARIUM",
 'WILDERNESS PARADISE EDUCATIONAL & ZOOLOGICAL PARK',
 "OGLEBAY'S GOOD ZOO",
 'POCATELLO ZOOLOGICAL SOCIETY',
 'MONTGOMERY AREA ZOOLOGICAL SOCIETY',
 'REVAS PETTING ZOO',
 'NAPLES ZOO AT CARIBBEAN GARDENS']

## Step 4: Sanity check — false-positive risk

Confirm Kalamazoo entries don't sneak in (regex word-boundary should prevent this), and that we aren't pulling in academic "Museum of Zoology" type entries that happen to be miscategorized.

In [5]:
# Any Kalamazoo false positives?
kalamazoo_in_zoos = zoos["Museum Name"].str.contains("kalamazoo", case=False).sum()
print(f"Kalamazoo false positives in zoo set: {kalamazoo_in_zoos}")

# Any academic 'museum of zoology' type entries in our zoo set?
academic = zoos["Museum Name"].str.contains(
    r"museum.*zool|zool.*museum", case=False, regex=True)
print(f"Academic 'zoology museum' entries in zoo set: {academic.sum()}")

Kalamazoo false positives in zoo set: 0
Academic 'zoology museum' entries in zoo set: 1


## Step 5: Final answer

In [6]:
zoo_count = len(zoos)
print("=" * 50)
print(f"  TOTAL ZOOS IN THE DATASET:  {zoo_count}")
print("=" * 50)

  TOTAL ZOOS IN THE DATASET:  258


## Summary

**There are 258 zoos in the dataset.**

This count comes from intersecting two filters:
- IMLS classifies the museum as `ZOO, AQUARIUM, OR WILDLIFE CONSERVATION` (564 records)
- The museum's name contains the word "zoo" (`\bzoo` regex, case-insensitive)

Why intersect rather than use just one? The IMLS category alone (564) bundles in aquariums and wildlife sanctuaries. A name-only filter (278 dataset-wide) picks up academic zoology museums and other non-zoo entities miscategorized in other buckets. The intersection (258) is the tightest defensible count of actual zoos.